# 駕駛模型實驗室 · DGX Spark

**影像取得 → 定位／資料檢查 → 訓練視覺化 → 驗證 → ONNX → Chestnut 車端測試**

先在本目錄執行 `bash bootstrap.sh`，並選擇 **駕駛模型實驗室 · DGX Spark** kernel。
這份 Notebook 使用你自己的實驗資料；尚未取得資料的圖表會留空。車端部署需另依部署手冊操作，這裡不會啟用車輛控制。

| 位置 | 任務 |
|---|---|
| Comma 3X／four | 記錄原始雙鏡頭、車速、姿態與 rlog |
| DGX Spark | 資料整理、訓練、模擬、匯出 |
| Comma 3X＋Chestnut | 候選模型編譯、停車測試、封閉場地驗證 |


In [ ]:
from pathlib import Path
import sys, subprocess, json
from IPython.display import display, Markdown
LAB = Path.cwd()
if not (LAB / "lab.py").exists():
    raise RuntimeError("請從 tools/distill_lab 目錄啟動 Jupyter")
sys.path.insert(0, str(LAB / "upstream"))
sys.path.insert(0, str(LAB))
WORK = LAB / "work"
RUN = "ev6-001"
def run(*arguments):
    result = subprocess.run([sys.executable, str(LAB / "lab.py"), "--work", str(WORK), *map(str, arguments)], text=True, capture_output=True)
    print(result.stdout)
    if result.stderr: print(result.stderr)
    if result.returncode: raise RuntimeError(f"流程停止，exit={result.returncode}；請處理上方錯誤")
run("doctor")

## 1 · 取得影像

先在 Spark 設定 `comma` SSH 別名與你自己的裝置金鑰。`ROUTE` 是裝置資料目錄名稱移除最後的 `--片段序號`。若已有原始檔，可用手冊的 `prepare --source`。

**只拍到相同道路不代表兩台裝置內參相同。**匯入時會核對真實 deviceType、sensor、解析度、校正與雙鏡頭時間，不能用重新縮放過的影片取代原始資料。

In [ ]:
HOST = "comma"
ROUTE = ""  # 填入實際行程，不能填範例文字
if not ROUTE:
    print("尚未指定行程；先在終端機執行 ssh comma 'ls /data/media/0/realdata'")
else:
    run("fetch-device", "--host", HOST, "--route", ROUTE)

### 可選：先用公開資料做硬體 smoke 測試

本 cell 預設不下載；填入 ID 後執行。缺少完整定位／frame_info 的資料版本會明確停止。單片段只能驗證程式，不能驗證模型泛化。

In [ ]:
PUBLIC_SEGMENTS = []  # 例如 ["2333e255f1de1fe83ea5975b24a3cc67"]
if PUBLIC_SEGMENTS:
    run("fetch-public", "--segments", *PUBLIC_SEGMENTS)
else:
    print("未選公開片段；可以直接使用自己的行車資料。")

## 2 · 定位與檢查

`raw-gnss` 要求有效 deviceMotion、外接 GPS 與原始 UBlox。只有真的存在有效 ECEF 濾波狀態時，才可明確改用 `logged-ecef`，且其標籤品質會分開記錄。失敗片段保留原檔與錯誤原因。

In [ ]:
METHOD = "raw-gnss"
if ROUTE:
    run("prepare-route", "--route", ROUTE, "--method", METHOD)
run("inventory")

In [ ]:
import ipywidgets as widgets
from drive_lab.visualization import segment_preview
segments = sorted(p.parent for p in (WORK / "data").glob("*/quality.json"))
if segments:
    chooser = widgets.Dropdown(options=[(p.name, str(p)) for p in segments], description="片段")
    output = widgets.Output()
    def preview(change=None):
        with output:
            output.clear_output(wait=True)
            segment_preview(chooser.value)
    chooser.observe(preview, names="value")
    display(chooser, output); preview()
else:
    print("尚無完成定位的個人片段；公開資料可在 evaluate 後查看模型裁切與軌跡。")

## 3 · 切分資料

同一趟行程的所有分鐘放在同一組。正式訓練至少兩趟行程；`SMOKE_ONLY=True` 只檢查硬體，沒有驗證組，不能匯出為已評估候選。
要學人工駕駛風格，保留 `ALLOW_ASSISTED=False`。

In [ ]:
SMOKE_ONLY = False
ALLOW_ASSISTED = False
arguments = ["split"]
if SMOKE_ONLY: arguments.append("--smoke")
if ALLOW_ASSISTED: arguments.append("--allow-assisted")
run(*arguments)

## 4 · 下載教師模型與視覺化訓練

教師版本已固定。下方按鈕會在本機啟動訓練；先從 batch 1 和短流程量測實際耗時。停止後可從最後已保存的 checkpoint 續訓。

In [ ]:
if not (WORK / "models" / "big_driving_supercombo.onnx").exists():
    run("fetch-teacher")
else:
    print("教師模型已存在。")
from drive_lab.visualization import panel
panel(WORK)

### TensorBoard 的即時曲線與影像

另開 Spark 終端機：
```bash
upstream/.venv/bin/tensorboard --logdir work/runs --host 127.0.0.1 --port 6006
```
經 SSH forwarding 後，在自己的瀏覽器開啟 http://127.0.0.1:6006 。資料預設保留本地。


## 5 · 驗證與真實軌跡對照

`RUN` 必須與上方訓練面板的實驗名稱相同。`best.pt` 依驗證集軌跡 ADE 選取。這是 open-loop 比較，不能直接判斷模型控制後能否恢復偏差。

In [ ]:
RUN = "ev6-001"
run("evaluate", "--name", RUN, "--checkpoint", "best.pt")
from drive_lab.visualization import training_curves, compare_example
training_curves(WORK / "runs" / RUN)
compare_example(WORK / "runs" / RUN)
display(Markdown("```json\n" + (WORK / "runs" / RUN / "evaluation.json").read_text() + "\n```"))

## 6 · 可選世界模型／DAgger 微調

在 Spark 終端機執行：
```bash
upstream/.venv/bin/python lab.py worldmodel-download
upstream/.venv/bin/python lab.py worldmodel-server
```
另開有圖形桌面的終端機執行 `simulate`，原生 Raylib 視窗會顯示模擬。詳細命令與 `rl-train`、RL 權重重新評估方式見 `docs/WORKFLOW_zh-TW.md` 第 7 節。

世界模型伺服器在 Spark 的完整 GPU 路徑仍需實測。RL 訓練使用訓練組片段，模擬評估使用未參與該次訓練的片段。

## 7 · 匯出車端候選

每次 release 使用新名稱。重新評估後的相同 checkpoint 才能匯出，輸出包含數值比對與來源 hash。若要採用 RL，須先 evaluate 相同 `--on-policy`，再加入同名參數匯出。

In [ ]:
RELEASE = "ev6-r001"
# 執行此 cell 會建立候選檔，不會複製到車上或啟用車輛控制。
run("export", "--name", RUN, "--release", RELEASE, "--checkpoint", "best.pt")
display(Markdown("候選模型路徑：`" + str(WORK / "releases" / RELEASE) + "`"))

## 8 · 車端部署與回復

開啟 [部署手冊](docs/DEPLOYMENT_zh-TW.md)。流程是：

1. rsync 候選 bundle 到裝置暫存區。
2. `car.py stage` 核對 checksum。
3. 停車／offroad 下由 Chestnut 執行 `compile`、`benchmark`。
4. 確認軟體契約、模型選單及評估結果，再進行封閉場地候選測試。
5. `activate --closed-course` 先完整備份，再安裝；不自動重啟。
6. 異常時 `restore --backup <實際備份路徑>` 回復。

**ONNX 匯出成功不等於能在公共道路安全駕駛。**目前尚未實測你的 Spark、EV6 rlog 或 Chestnut，第一次真實資料執行結果會決定還需要哪些轉接修正。